# Task 1: Structured LLM Prompt Pipeline & JSON Extractor
**Goal:** Extract structured data from unstructured text using LLM APIs (Gemini or OpenAI) with Python.

**Pipeline:** parameterized prompt template → few-shot example → LLM call (JSON mode) → Pydantic validation → auto-retry with error feedback → saved JSON.

**Supports:** invoices and resumes (add more in the `DOC_TYPES` registry).

**Setup:** add your API key in Colab → 🔑 *Secrets* (left sidebar): `GOOGLE_API_KEY` (Gemini) or `OPENAI_API_KEY` (OpenAI). Enable "Notebook access".

In [1]:
!pip install -q google-genai openai pydantic

In [2]:
import os, re, json, time
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

# ---------- CONFIG ----------
PROVIDER = "gemini"            # "gemini" or "openai"
GEMINI_MODEL = "gemini-3.8-flash"   # if 404 error: change to the model name shown in the error message
OPENAI_MODEL = "gpt-4o-mini"
MAX_RETRIES = 3                # validation retries per document
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_key(name: str) -> str:
    """Read key from env, then Colab Secrets, then prompt (never hard-code keys)."""
    key = os.environ.get(name)
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    from getpass import getpass
    return getpass(f"Enter {name}: ")

## 1. Pydantic schemas (strict validation)

In [3]:
DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
MONTH_RE = re.compile(r"^(\d{4}-\d{2}|present)$", re.I)
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


class LineItem(BaseModel):
    description: str
    quantity: float = Field(ge=0)
    unit_price: float = Field(ge=0)
    amount: float = Field(ge=0)


class Invoice(BaseModel):
    invoice_number: str
    invoice_date: str = Field(description="YYYY-MM-DD")
    due_date: Optional[str] = Field(default=None, description="YYYY-MM-DD or null")
    vendor_name: str
    customer_name: str
    currency: str = Field(min_length=3, max_length=3, description="ISO code, e.g. USD")
    line_items: List[LineItem] = Field(min_length=1)
    subtotal: float = Field(ge=0)
    tax: float = Field(default=0.0, ge=0)
    total: float = Field(ge=0)

    @field_validator("invoice_date", "due_date")
    @classmethod
    def _date_fmt(cls, v):
        if v is not None and not DATE_RE.match(v):
            raise ValueError("date must be YYYY-MM-DD")
        return v

    @field_validator("currency")
    @classmethod
    def _cur(cls, v):
        return v.upper()

    @model_validator(mode="after")
    def _totals(self):
        if abs(self.subtotal + self.tax - self.total) > 0.01:
            raise ValueError(f"subtotal + tax ({self.subtotal + self.tax}) must equal total ({self.total})")
        return self


class Experience(BaseModel):
    company: str
    title: str
    start_date: str = Field(description="YYYY-MM")
    end_date: str = Field(description="YYYY-MM or 'present'")
    highlights: List[str] = []

    @field_validator("start_date", "end_date")
    @classmethod
    def _m(cls, v):
        if not MONTH_RE.match(v):
            raise ValueError("must be YYYY-MM or 'present'")
        return v


class Education(BaseModel):
    institution: str
    degree: str
    graduation_year: Optional[int] = Field(default=None, ge=1950, le=2100)


class Resume(BaseModel):
    full_name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    location: Optional[str] = None
    summary: Optional[str] = None
    skills: List[str] = Field(min_length=1)
    experience: List[Experience] = []
    education: List[Education] = []

    @field_validator("email")
    @classmethod
    def _email(cls, v):
        if v is not None and not EMAIL_RE.match(v):
            raise ValueError("invalid email")
        return v

## 2. Parameterized prompt templates + few-shot examples

In [4]:
SYSTEM_PROMPT = (
    "You are a precise information-extraction engine. "
    "Return ONLY one valid JSON object that matches the given schema. "
    "No markdown, no commentary. Use null for missing optional fields. "
    "Never invent data that is not in the document. Numbers must be plain numbers (no currency symbols)."
)

PROMPT_TEMPLATE = """{instructions}

### JSON SCHEMA
{schema}

### EXAMPLE
Input:
{example_input}

Output:
{example_output}

### DOCUMENT
{document}

### OUTPUT (JSON only)"""

INVOICE_EX_IN = """INVOICE #A-1001
Date: March 5, 2024   Due: April 4, 2024
From: Bright Supplies Ltd.
Bill to: Kiran Traders
2 x Office Chair @ $50.00 = $100.00
Tax (10%): $10.00
Total: $110.00"""
INVOICE_EX_OUT = {
    "invoice_number": "A-1001", "invoice_date": "2024-03-05", "due_date": "2024-04-04",
    "vendor_name": "Bright Supplies Ltd.", "customer_name": "Kiran Traders", "currency": "USD",
    "line_items": [{"description": "Office Chair", "quantity": 2, "unit_price": 50.0, "amount": 100.0}],
    "subtotal": 100.0, "tax": 10.0, "total": 110.0,
}

RESUME_EX_IN = """Priya Sharma | priya@example.com | +91 98765 43210 | Pune, India
Data analyst with 3 years of experience.
Skills: Python, SQL, Tableau
Acme Analytics - Data Analyst (Jan 2021 - Present): Built dashboards used by 40 managers.
B.Tech Computer Science, IIT Pune, 2020"""
RESUME_EX_OUT = {
    "full_name": "Priya Sharma", "email": "priya@example.com", "phone": "+91 98765 43210",
    "location": "Pune, India", "summary": "Data analyst with 3 years of experience.",
    "skills": ["Python", "SQL", "Tableau"],
    "experience": [{"company": "Acme Analytics", "title": "Data Analyst", "start_date": "2021-01",
                    "end_date": "present", "highlights": ["Built dashboards used by 40 managers"]}],
    "education": [{"institution": "IIT Pune", "degree": "B.Tech Computer Science", "graduation_year": 2020}],
}

DOC_TYPES = {
    "invoice": {
        "model": Invoice,
        "instructions": "Extract invoice data. Dates must be ISO (YYYY-MM-DD). Currency is a 3-letter ISO code.",
        "example_input": INVOICE_EX_IN,
        "example_output": json.dumps(INVOICE_EX_OUT, indent=2),
    },
    "resume": {
        "model": Resume,
        "instructions": "Extract resume data. Experience dates must be YYYY-MM, use 'present' for current jobs.",
        "example_input": RESUME_EX_IN,
        "example_output": json.dumps(RESUME_EX_OUT, indent=2),
    },
}

# Sanity check: few-shot examples must satisfy their own schemas
for name, cfg in DOC_TYPES.items():
    cfg["model"].model_validate_json(cfg["example_output"])
print("Few-shot examples valid ✅")


def build_prompt(doc_type: str, document: str) -> str:
    cfg = DOC_TYPES[doc_type]
    return PROMPT_TEMPLATE.format(
        instructions=cfg["instructions"],
        schema=json.dumps(cfg["model"].model_json_schema(), indent=2),
        example_input=cfg["example_input"],
        example_output=cfg["example_output"],
        document=document.strip(),
    )

Few-shot examples valid ✅


## 3. LLM connectors (Gemini / OpenAI) in JSON mode

In [5]:
_client = {}


def call_llm(prompt: str, retries: int = 2) -> str:
    for attempt in range(retries + 1):
        try:
            if PROVIDER == "gemini":
                from google import genai
                from google.genai import types
                if "gemini" not in _client:
                    _client["gemini"] = genai.Client(api_key=get_key("GOOGLE_API_KEY"))
                r = _client["gemini"].models.generate_content(
                    model=GEMINI_MODEL,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        response_mime_type="application/json",
                        temperature=0,
                    ),
                )
                return r.text
            else:
                from openai import OpenAI
                if "openai" not in _client:
                    _client["openai"] = OpenAI(api_key=get_key("OPENAI_API_KEY"))
                r = _client["openai"].chat.completions.create(
                    model=OPENAI_MODEL, temperature=0,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": SYSTEM_PROMPT},
                              {"role": "user", "content": prompt}],
                )
                return r.choices[0].message.content
        except Exception as e:  # network / rate-limit
            if attempt == retries:
                raise
            print(f"API error ({e}); retrying...")
            time.sleep(2 ** attempt)


def clean_json(text: str) -> str:
    """Strip code fences / stray text around the JSON object."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    s, e = text.find("{"), text.rfind("}")
    return text[s:e + 1] if s != -1 and e != -1 else text

## 4. Extraction pipeline with validation + self-correcting retries

In [6]:
def extract(doc_type: str, document: str) -> BaseModel:
    model_cls = DOC_TYPES[doc_type]["model"]
    base_prompt = build_prompt(doc_type, document)
    error = None
    for attempt in range(1, MAX_RETRIES + 1):
        prompt = base_prompt if error is None else (
            base_prompt + f"\n\nYour previous output failed validation:\n{error}\n"
            "Fix these problems and return the corrected JSON only.")
        raw = call_llm(prompt)
        try:
            result = model_cls.model_validate_json(clean_json(raw))
            print(f"✅ {doc_type}: valid on attempt {attempt}")
            return result
        except ValueError as e:  # includes pydantic.ValidationError
            error = str(e)
            print(f"⚠️ {doc_type}: attempt {attempt} invalid -> retrying")
    raise RuntimeError(f"Could not get valid {doc_type} JSON after {MAX_RETRIES} tries:\n{error}")


def extract_and_save(doc_type: str, document: str, filename: str) -> dict:
    data = extract(doc_type, document).model_dump()
    path = f"{OUTPUT_DIR}/{filename}"
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved -> {path}")
    return data

## 5. Run on sample unstructured documents

In [7]:
SAMPLE_INVOICE = """
Tax Invoice No. INV/2024/0457
Issued 12 Jan 2024, payment due within 30 days (by 11 Feb 2024).
Seller: GreenLeaf Stationers Pvt Ltd, Mumbai
Buyer: Rohan Enterprises
Items:
 - A4 Paper Ream, qty 10, Rs 250 each, Rs 2,500
 - Ballpoint Pens (box), qty 5, Rs 120 each, Rs 600
Subtotal Rs 3,100 | GST 18%: Rs 558 | Grand total: Rs 3,658
All amounts in INR.
"""

SAMPLE_RESUME = """
Arjun Mehta - Backend Developer
arjun.mehta@mail.com, 9876501234, Bengaluru
I build scalable APIs and love clean code.
Worked at Zoho as Software Engineer from June 2020 to Aug 2022 (built billing microservices, cut latency 35%).
Currently at Swiggy since Sept 2022 as Senior Engineer - leads a 5 person team.
Tech: Python, Django, PostgreSQL, Docker, AWS
Education: B.E. Information Technology, Anna University, graduated 2020
"""

invoice_json = extract_and_save("invoice", SAMPLE_INVOICE, "invoice.json")
print(json.dumps(invoice_json, indent=2))

✅ invoice: valid on attempt 1
Saved -> /content/outputs/invoice.json
{
  "invoice_number": "INV/2024/0457",
  "invoice_date": "2024-01-12",
  "due_date": "2024-02-11",
  "vendor_name": "GreenLeaf Stationers Pvt Ltd",
  "customer_name": "Rohan Enterprises",
  "currency": "INR",
  "line_items": [
    {
      "description": "A4 Paper Ream",
      "quantity": 10.0,
      "unit_price": 250.0,
      "amount": 2500.0
    },
    {
      "description": "Ballpoint Pens (box)",
      "quantity": 5.0,
      "unit_price": 120.0,
      "amount": 600.0
    }
  ],
  "subtotal": 3100.0,
  "tax": 558.0,
  "total": 3658.0
}


In [8]:
resume_json = extract_and_save("resume", SAMPLE_RESUME, "resume.json")
print(json.dumps(resume_json, indent=2))

✅ resume: valid on attempt 1
Saved -> /content/outputs/resume.json
{
  "full_name": "Arjun Mehta",
  "email": "arjun.mehta@mail.com",
  "phone": "9876501234",
  "location": "Bengaluru",
  "summary": "I build scalable APIs and love clean code.",
  "skills": [
    "Python",
    "Django",
    "PostgreSQL",
    "Docker",
    "AWS"
  ],
  "experience": [
    {
      "company": "Swiggy",
      "title": "Senior Engineer",
      "start_date": "2022-09",
      "end_date": "present",
      "highlights": [
        "leads a 5 person team"
      ]
    },
    {
      "company": "Zoho",
      "title": "Software Engineer",
      "start_date": "2020-06",
      "end_date": "2022-08",
      "highlights": [
        "built billing microservices, cut latency 35%"
      ]
    }
  ],
  "education": [
    {
      "institution": "Anna University",
      "degree": "B.E. Information Technology",
      "graduation_year": 2020
    }
  ]
}


## 6. Prove the validation works (no API needed)

In [9]:
bad = '{"invoice_number":"X1","invoice_date":"05/03/2024","vendor_name":"A","customer_name":"B","currency":"USD","line_items":[],"subtotal":100,"tax":10,"total":500}'
try:
    Invoice.model_validate_json(bad)
except ValidationError as e:
    print("Correctly rejected invalid JSON:\n", e)

Correctly rejected invalid JSON:
 2 validation errors for Invoice
invoice_date
  Value error, date must be YYYY-MM-DD [type=value_error, input_value='05/03/2024', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error
line_items
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/too_short


## 7. (Optional) Try your own document

In [11]:
my_doc = input("Paste any invoice text (or press Enter to skip): ")
if my_doc.strip():
    print(json.dumps(extract("invoice", my_doc).model_dump(), indent=2))

Paste any invoice text (or press Enter to skip): 
